# Preliminary Paper Runbook (Colab)

This notebook is the resumable Colab entrypoint for fast preliminary paper iteration.

What it does:
1. Mounts Google Drive and resolves the repo root.
2. Creates a run-specific runtime config under `runs/experiments/<run_id>/configs/`.
3. Uses a stage manifest so completed expensive stages are skipped on rerun.
4. Tracks ETA per stage and shows remaining ETA before and after execution.
5. Runs each major stage in its own execution cell so a disconnect does not throw away unrelated progress.
6. Benchmarks each mode in its own tracked step, then merges the per-mode outputs into the final benchmark artifacts.

Profiles:
- `RUN_PROFILE = "fast"` keeps the default path practical for Colab reruns.
- `RUN_PROFILE = "full"` restores the broader paper pipeline.

Resume behavior:
- If a tracked stage already completed and its outputs still exist, rerunning that stage cell skips it.
- Benchmark mode runs checkpoint independently, so a disconnect during `backprop_contrastive` does not invalidate finished FF modes.
- If a stage fails, fix the issue and rerun only the relevant cell.


In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = None
candidates = [
    Path(os.environ.get("FRM_REPO_DIR", "")).expanduser() if os.environ.get("FRM_REPO_DIR") else None,
    Path.cwd(),
    Path("/content/Forward-Risk-Manager"),
    Path("/content/drive/MyDrive/Forward-Risk-Manager"),
    Path("/content/drive/MyDrive/forward-risk-manager"),
]
for candidate in candidates:
    if candidate is None:
        continue
    if (candidate / "configs" / "default.toml").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError("Could not locate repo root containing configs/default.toml")

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from frisk.notebook_runtime import (
    NotebookStage,
    format_duration,
    merge_csv_files,
    record_run_step,
    remaining_eta_seconds,
    run_tracked_command,
    shell_quote,
    stage_status_rows,
    step_is_complete,
    write_toml_overrides,
)

print("repo root:", ROOT)
print("cwd:", Path.cwd())


In [ ]:
from datetime import datetime, timezone
import importlib.util
import json
import re

import pandas as pd

PYTHON_EXE = shell_quote(sys.executable)

RUN_PROFILE = "fast"  # "fast" | "full"
RUN_ID_OVERRIDE = ""
RESUME_POLICY = "auto"  # "auto" | "force_new" | "force_resume"
DEVICE = "cuda"
INSTALL_DEPS = IN_COLAB

if RUN_PROFILE not in {"fast", "full"}:
    raise ValueError(f"Unsupported RUN_PROFILE: {RUN_PROFILE}")

PROFILE_STAGE_DEFAULTS = {
    "fast": {
        "RUN_BUILD_GRAPHS": False,
        "RUN_BENCHMARK": True,
        "RUN_TRAIN": True,
        "RUN_SWEEP": False,
        "RUN_DUAL_SCORE": False,
        "RUN_SCENARIO": False,
        "RUN_BACKTEST": False,
        "RUN_PUBLISH": False,
    },
    "full": {
        "RUN_BUILD_GRAPHS": False,
        "RUN_BENCHMARK": True,
        "RUN_TRAIN": True,
        "RUN_SWEEP": True,
        "RUN_DUAL_SCORE": True,
        "RUN_SCENARIO": True,
        "RUN_BACKTEST": True,
        "RUN_PUBLISH": False,
    },
}
STAGE_DEFAULTS = PROFILE_STAGE_DEFAULTS[RUN_PROFILE].copy()
RUN_BUILD_GRAPHS = STAGE_DEFAULTS["RUN_BUILD_GRAPHS"]
RUN_BENCHMARK = STAGE_DEFAULTS["RUN_BENCHMARK"]
RUN_TRAIN = STAGE_DEFAULTS["RUN_TRAIN"]
RUN_SWEEP = STAGE_DEFAULTS["RUN_SWEEP"]
RUN_DUAL_SCORE = STAGE_DEFAULTS["RUN_DUAL_SCORE"]
RUN_SCENARIO = STAGE_DEFAULTS["RUN_SCENARIO"]
RUN_BACKTEST = STAGE_DEFAULTS["RUN_BACKTEST"]
RUN_PUBLISH = STAGE_DEFAULTS["RUN_PUBLISH"]

BASE_CONFIG = ROOT / "configs" / "paper_final_500.toml"
DEFAULT_CONFIG = ROOT / "configs" / "default.toml"
PREBUILT_GRAPH_PATH = ROOT / "data" / "processed" / "graphs_master_ff_rich.pt"
SHARDED_GRAPH_PATH = Path(str(PREBUILT_GRAPH_PATH) + ".sharded")

DEFAULT_BENCHMARK_MODES = [
    "ff_layerwise",
    "ff_e2e",
    "backprop_contrastive",
    "backprop_supervised_return",
]
FAST_BENCHMARK_MODES = ["ff_layerwise", "ff_e2e", "backprop_contrastive"]
BENCHMARK_MODES = FAST_BENCHMARK_MODES.copy() if RUN_PROFILE == "fast" else DEFAULT_BENCHMARK_MODES.copy()
BENCHMARK_MODES = [str(mode).strip() for mode in BENCHMARK_MODES if str(mode).strip()]
BENCHMARK_MODES = list(dict.fromkeys(BENCHMARK_MODES)) or DEFAULT_BENCHMARK_MODES.copy()

FAST_TRAIN_OVERRIDES = {
    "epochs": 60,
    "graph_limit": 0,
    "graph_limit_keep_recent": True,
    "epoch_graph_fraction": 0.25,
    "epoch_graph_min": 1024,
    "epoch_graph_mode": "recent_bias",
    "torch_compile": False,
    "auto_tune_batch": True,
}
FAST_BENCHMARK_OVERRIDES = {
    "epochs": 15,
    "walk_forward_max_folds": 2,
    "eval_neg_modes": ["time_flip"],
    "timing_warmup_epochs": 1,
}
FAST_SWEEP_OVERRIDES = {
    "max_runs": 8,
    "walk_forward_max_folds_cap": 2,
    "eval_neg_modes": ["time_flip"],
}

PRICE_CANDIDATES = [
    ROOT / "data" / "processed" / "prices.csv",
    ROOT / "data" / "consolidated_ff_local" / "prices.csv",
    ROOT / "data" / "processed_long" / "prices.csv",
]
CONSTITUENT_CANDIDATES = [
    ROOT / "data" / "processed" / "constituents.csv",
    ROOT / "data" / "processed_long" / "constituents.csv",
]
MACRO_CANDIDATES = [
    ROOT / "data" / "processed" / "macro.csv",
    ROOT / "data" / "consolidated_ff_local" / "macro.csv",
]


def first_existing(paths, *, required=True):
    for path in paths:
        if path.exists():
            return path.resolve()
    if required:
        raise FileNotFoundError("No existing path found in: " + ", ".join(str(p) for p in paths))
    return None


def resolve_graph_source_path(base_path: Path) -> tuple[Path, str]:
    sharded_path = Path(str(base_path) + ".sharded")
    manifest_path = sharded_path / "manifest.json"
    if sharded_path.is_dir() and manifest_path.exists():
        return sharded_path.resolve(), "sharded"
    if base_path.exists():
        return base_path.resolve(), "packed"
    raise FileNotFoundError(
        f"Missing graph artifact. Checked packed={base_path} and sharded={sharded_path}"
    )


def estimate_walk_forward_folds(
    n_items: int | None,
    *,
    train_frac: float,
    eval_frac: float,
    step_frac: float,
    min_train: int,
    min_eval: int,
    max_folds: int,
) -> int | None:
    if n_items is None or n_items < 2:
        return None
    train_size = min(max(max(int(round(n_items * float(train_frac))), int(min_train)), 1), n_items - 1)
    eval_size = min(max(max(int(round(n_items * float(eval_frac))), int(min_eval)), 1), n_items - train_size)
    if eval_size <= 0:
        return 0
    step_size = max(1, int(round(n_items * float(step_frac)))) if float(step_frac) > 0 else eval_size
    folds = 0
    eval_start = train_size
    while eval_start + eval_size <= n_items:
        folds += 1
        if max_folds > 0 and folds >= int(max_folds):
            break
        eval_start += step_size
    return folds


def load_toml_section(path: Path, section: str) -> dict:
    try:
        import tomllib
    except ModuleNotFoundError:
        import tomli as tomllib
    with path.open("rb") as handle:
        payload = tomllib.load(handle)
    values = payload.get(section, {})
    return dict(values) if isinstance(values, dict) else {}


def mode_slug(mode: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(mode).strip()).strip("_") or "mode"


DATA_PRICES_PATH = first_existing(PRICE_CANDIDATES, required=True)
DATA_CONSTITUENTS_PATH = first_existing(CONSTITUENT_CANDIDATES, required=False)
DATA_MACRO_PATH = first_existing(MACRO_CANDIDATES, required=False)

if RESUME_POLICY == "force_resume" and not RUN_ID_OVERRIDE.strip():
    raise ValueError("RESUME_POLICY='force_resume' requires RUN_ID_OVERRIDE")

resume_requested = bool(RUN_ID_OVERRIDE.strip()) and RESUME_POLICY != "force_new"
if resume_requested:
    RUN_ID = RUN_ID_OVERRIDE.strip()
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID
else:
    RUN_ID = f"paper_prelim_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID

for sub in ("configs", "data", "metrics", "plots", "logs", "models", "diagnostics"):
    (RUN_ROOT / sub).mkdir(parents=True, exist_ok=True)

RUNTIME_CONFIG = RUN_ROOT / "configs" / "runtime_config.toml"
MODE_CONFIG_DIR = RUN_ROOT / "configs" / "benchmark_modes"
MODE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = RUN_ROOT / "logs" / "stage_manifest.json"
LOG_DIR = RUN_ROOT / "logs" / "stage_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_OUT = RUN_ROOT / "data" / "graphs_preliminary.pt"
GRAPH_RESOLVED_PATH = GRAPH_OUT.resolve() if RUN_BUILD_GRAPHS else resolve_graph_source_path(PREBUILT_GRAPH_PATH)[0]
GRAPH_FORMAT = "packed" if RUN_BUILD_GRAPHS else resolve_graph_source_path(PREBUILT_GRAPH_PATH)[1]
GRAPH_PATH_FOR_RUN = GRAPH_OUT if RUN_BUILD_GRAPHS else PREBUILT_GRAPH_PATH

TRAIN_LOG_CSV = RUN_ROOT / "metrics" / "ff_train.csv"
TRAIN_PLOT = RUN_ROOT / "plots" / "ff_train.png"
MODEL_CKPT = RUN_ROOT / "models" / "ff_model.pt"
ENCODER_CKPT = RUN_ROOT / "models" / "encoder.pt"
CRITIC_CKPT = RUN_ROOT / "models" / "critic.pt"
TRAIN_RESUME_STATE = RUN_ROOT / "models" / "train_resume.pt"

BENCHMARK_CSV = RUN_ROOT / "metrics" / "benchmark.csv"
BENCHMARK_FOLDS_CSV = RUN_ROOT / "metrics" / "benchmark_walk_forward_folds.csv"
BENCHMARK_BASELINE_CSV = RUN_ROOT / "metrics" / "benchmark_baseline.csv"
BENCHMARK_HISTORY_CSV = RUN_ROOT / "metrics" / "benchmark_history.csv"
BENCHMARK_PLOT = RUN_ROOT / "plots" / "benchmark_speed_sep.png"
BENCHMARK_BAR = RUN_ROOT / "plots" / "benchmark.png"
PAPER_SUMMARY_MD = RUN_ROOT / "logs" / "paper_benchmark_summary.md"
PAPER_SUMMARY_CSV = RUN_ROOT / "metrics" / "paper_benchmark_summary.csv"
PAPER_SUMMARY_JSON = RUN_ROOT / "logs" / "paper_benchmark_summary.json"

MODE_METRICS_DIR = RUN_ROOT / "metrics" / "benchmark_modes"
MODE_PLOTS_DIR = RUN_ROOT / "plots" / "benchmark_modes"
BENCHMARK_RESUME_DIR = RUN_ROOT / "models" / "benchmark_resume"
MODE_METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODE_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_RESUME_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_MODE_SPECS = []
for mode in BENCHMARK_MODES:
    slug = mode_slug(mode)
    BENCHMARK_MODE_SPECS.append(
        {
            "mode": mode,
            "slug": slug,
            "step": f"benchmark_{slug}",
            "config_path": MODE_CONFIG_DIR / f"benchmark_{slug}.toml",
            "out_csv": MODE_METRICS_DIR / f"benchmark_{slug}.csv",
            "folds_csv": MODE_METRICS_DIR / f"benchmark_walk_forward_folds_{slug}.csv",
            "baseline_csv": MODE_METRICS_DIR / f"benchmark_baseline_{slug}.csv",
            "history_csv": MODE_METRICS_DIR / f"benchmark_history_{slug}.csv",
            "plot_path": MODE_PLOTS_DIR / f"benchmark_speed_sep_{slug}.png",
            "bar_plot_path": MODE_PLOTS_DIR / f"benchmark_bar_{slug}.png",
        }
    )

SWEEP_CSV = RUN_ROOT / "metrics" / "ff_sweep.csv"
DUAL_SCORE_TXT = RUN_ROOT / "logs" / "dual_score_report.txt"
DUAL_SCORE_CSV = RUN_ROOT / "metrics" / "dual_score_report.csv"

SCENARIO_CSV = RUN_ROOT / "metrics" / "scenario_book.csv"
SCENARIO_DIAG_CSV = RUN_ROOT / "diagnostics" / "scenario_constraint_diagnostics.csv"
STRESS_CSV = RUN_ROOT / "metrics" / "stress_test_report.csv"
STRESS_PLOT = RUN_ROOT / "plots" / "stress_test_report.png"
CALIBRATION_JSON = RUN_ROOT / "diagnostics" / "hallucination_calibration.json"
CALIBRATION_BY_TICKER_CSV = RUN_ROOT / "diagnostics" / "hallucination_calibration_by_ticker.csv"

GOODNESS_CSV = RUN_ROOT / "diagnostics" / "goodness_backtest.csv"
GOODNESS_QUANTILES_CSV = RUN_ROOT / "diagnostics" / "goodness_quantiles.csv"
GOODNESS_PLOT = RUN_ROOT / "plots" / "goodness_scatter.png"
GOODNESS_EVENTS_CSV = RUN_ROOT / "diagnostics" / "goodness_events.csv"
GOODNESS_STRATEGY_CSV = RUN_ROOT / "diagnostics" / "goodness_strategy_metrics.csv"
GOODNESS_TIMELINE_PLOT = RUN_ROOT / "plots" / "goodness_timeline.png"

section_overrides = {
    "build_graphs": {
        "prices": str(DATA_PRICES_PATH),
        "out": str(GRAPH_OUT),
    },
    "train": {
        "graphs": str(GRAPH_PATH_FOR_RUN),
        "device": DEVICE,
        "log_csv": str(TRAIN_LOG_CSV),
        "plot_path": str(TRAIN_PLOT),
        "save_model": str(MODEL_CKPT),
        "save_encoder": str(ENCODER_CKPT),
        "save_critic": str(CRITIC_CKPT),
        "resume_enabled": True,
        "resume_state_path": str(TRAIN_RESUME_STATE),
        "resume_save_every_epochs": 1,
    },
    "benchmark": {
        "out_csv": str(BENCHMARK_CSV),
        "walk_forward_out_csv": str(BENCHMARK_FOLDS_CSV),
        "baseline_out_csv": str(BENCHMARK_BASELINE_CSV),
        "history_out_csv": str(BENCHMARK_HISTORY_CSV),
        "plot_path": str(BENCHMARK_PLOT),
        "bar_plot_path": str(BENCHMARK_BAR),
        "econ_prices": str(DATA_PRICES_PATH),
        "resume_enabled": True,
        "resume_dir": str(BENCHMARK_RESUME_DIR),
        "resume_save_every_epochs": 1,
    },
    "sweep": load_toml_section(DEFAULT_CONFIG, "sweep"),
    "scenario_book": load_toml_section(DEFAULT_CONFIG, "scenario_book"),
}
section_overrides["sweep"].update({
    "out_csv": str(SWEEP_CSV),
    "econ_prices": str(DATA_PRICES_PATH),
})
section_overrides["scenario_book"].update({
    "critic_model": str(CRITIC_CKPT),
    "diag_out": str(SCENARIO_DIAG_CSV),
    "out": str(SCENARIO_CSV),
})
if DATA_CONSTITUENTS_PATH is not None:
    section_overrides["build_graphs"]["constituents"] = str(DATA_CONSTITUENTS_PATH)
if DATA_MACRO_PATH is not None:
    section_overrides["build_graphs"]["macro"] = str(DATA_MACRO_PATH)

if RUN_PROFILE == "fast":
    section_overrides["train"].update(FAST_TRAIN_OVERRIDES)
    section_overrides["benchmark"].update(FAST_BENCHMARK_OVERRIDES)
    if RUN_SWEEP:
        section_overrides["sweep"].update(FAST_SWEEP_OVERRIDES)

write_toml_overrides(BASE_CONFIG, RUNTIME_CONFIG, section_overrides)

GRAPH_COUNT = None
GRAPH_NUM_SHARDS = None
GRAPH_SHARD_SIZE = None
GRAPH_MANIFEST_PATH = None
if GRAPH_FORMAT == "sharded":
    GRAPH_MANIFEST_PATH = GRAPH_RESOLVED_PATH / "manifest.json"
    manifest = json.loads(GRAPH_MANIFEST_PATH.read_text(encoding="utf-8"))
    GRAPH_COUNT = int(manifest.get("num_graphs", 0) or 0)
    GRAPH_NUM_SHARDS = int(manifest.get("num_shards", 0) or 0)
    GRAPH_SHARD_SIZE = int(manifest.get("shard_size", 0) or 0)

BENCHMARK_EPOCHS = int(section_overrides["benchmark"].get("epochs", FAST_BENCHMARK_OVERRIDES["epochs"] if RUN_PROFILE == "fast" else 500))
BENCHMARK_WALK_FORWARD_MAX_FOLDS = int(section_overrides["benchmark"].get("walk_forward_max_folds", 0))
BENCHMARK_FOLD_ESTIMATE = estimate_walk_forward_folds(
    GRAPH_COUNT,
    train_frac=0.6,
    eval_frac=0.2,
    step_frac=0.1,
    min_train=128,
    min_eval=32,
    max_folds=BENCHMARK_WALK_FORWARD_MAX_FOLDS,
)
BENCHMARK_MODE_COUNT = len(BENCHMARK_MODE_SPECS)
BENCHMARK_WORK_UNITS = (
    BENCHMARK_EPOCHS * BENCHMARK_MODE_COUNT * BENCHMARK_FOLD_ESTIMATE
    if BENCHMARK_FOLD_ESTIMATE is not None
    else None
)
BENCHMARK_TOTAL_ETA_MIN = 45 if RUN_PROFILE == "fast" else 150
BENCHMARK_MODE_ETA_S = (
    (BENCHMARK_TOTAL_ETA_MIN * 60) / max(1, BENCHMARK_MODE_COUNT)
    if BENCHMARK_MODE_COUNT
    else 0.0
)

ENABLED_STAGE_NAMES = [name for enabled, name in [(INSTALL_DEPS, "install")] if enabled]
if RUN_BUILD_GRAPHS:
    ENABLED_STAGE_NAMES.append("build_graphs")
if RUN_BENCHMARK:
    ENABLED_STAGE_NAMES.extend(spec["step"] for spec in BENCHMARK_MODE_SPECS)
    ENABLED_STAGE_NAMES.extend(["benchmark_aggregate", "paper_summary"])
if RUN_TRAIN:
    ENABLED_STAGE_NAMES.append("train")
if RUN_SWEEP:
    ENABLED_STAGE_NAMES.append("sweep")
if RUN_DUAL_SCORE and RUN_BENCHMARK and RUN_SWEEP:
    ENABLED_STAGE_NAMES.append("dual_score")
if RUN_SCENARIO and RUN_TRAIN:
    ENABLED_STAGE_NAMES.extend(["scenario", "stress", "calibration"])
if RUN_BACKTEST and RUN_TRAIN:
    ENABLED_STAGE_NAMES.append("backtest")
if RUN_PUBLISH:
    ENABLED_STAGE_NAMES.append("publish")

ETA_MIN = {
    "install": 8,
    "build_graphs": 60,
    "benchmark_total": BENCHMARK_TOTAL_ETA_MIN,
    "benchmark_aggregate": 2,
    "paper_summary": 5,
    "train": 75 if RUN_PROFILE == "fast" else 180,
    "sweep": 40 if RUN_PROFILE == "fast" else 240,
    "dual_score": 5,
    "scenario": 75,
    "stress": 5,
    "calibration": 3,
    "backtest": 20,
    "publish": 2,
}

STAGES = []
if INSTALL_DEPS:
    STAGES.append(NotebookStage("install", "Install dependencies", eta_s=ETA_MIN["install"] * 60))
if RUN_BUILD_GRAPHS:
    STAGES.append(NotebookStage("build_graphs", "Build graphs", (str(GRAPH_OUT),), eta_s=ETA_MIN["build_graphs"] * 60))
if RUN_BENCHMARK:
    for spec in BENCHMARK_MODE_SPECS:
        STAGES.append(
            NotebookStage(
                spec["step"],
                f"Benchmark {spec['mode']}",
                (
                    str(spec["out_csv"]),
                    str(spec["history_csv"]),
                    str(spec["plot_path"]),
                    str(spec["bar_plot_path"]),
                ),
                eta_s=BENCHMARK_MODE_ETA_S,
            )
        )
    STAGES.append(
        NotebookStage(
            "benchmark_aggregate",
            "Merge benchmark outputs",
            (
                str(BENCHMARK_CSV),
                str(BENCHMARK_FOLDS_CSV),
                str(BENCHMARK_BASELINE_CSV),
                str(BENCHMARK_HISTORY_CSV),
                str(BENCHMARK_PLOT),
                str(BENCHMARK_BAR),
            ),
            eta_s=ETA_MIN["benchmark_aggregate"] * 60,
        )
    )
    STAGES.append(NotebookStage("paper_summary", "Write paper benchmark summary", (str(PAPER_SUMMARY_MD), str(PAPER_SUMMARY_CSV), str(PAPER_SUMMARY_JSON)), eta_s=ETA_MIN["paper_summary"] * 60))
if RUN_TRAIN:
    STAGES.append(NotebookStage("train", "Train FF/BP model", (str(ENCODER_CKPT), str(CRITIC_CKPT)), eta_s=ETA_MIN["train"] * 60))
if RUN_SWEEP:
    STAGES.append(NotebookStage("sweep", "Run FF sweep", (str(SWEEP_CSV),), eta_s=ETA_MIN["sweep"] * 60))
if RUN_DUAL_SCORE and RUN_BENCHMARK and RUN_SWEEP:
    STAGES.append(NotebookStage("dual_score", "Write dual score report", (str(DUAL_SCORE_TXT), str(DUAL_SCORE_CSV)), eta_s=ETA_MIN["dual_score"] * 60))
if RUN_SCENARIO and RUN_TRAIN:
    STAGES.append(NotebookStage("scenario", "Run scenario book", (str(SCENARIO_CSV), str(SCENARIO_DIAG_CSV)), eta_s=ETA_MIN["scenario"] * 60, optional=True))
    STAGES.append(NotebookStage("stress", "Write stress report", (str(STRESS_CSV), str(STRESS_PLOT)), eta_s=ETA_MIN["stress"] * 60, optional=True))
    STAGES.append(NotebookStage("calibration", "Write hallucination calibration", (str(CALIBRATION_JSON), str(CALIBRATION_BY_TICKER_CSV)), eta_s=ETA_MIN["calibration"] * 60, optional=True))
if RUN_BACKTEST and RUN_TRAIN:
    STAGES.append(NotebookStage("backtest", "Run goodness backtest", (str(GOODNESS_CSV), str(GOODNESS_STRATEGY_CSV), str(GOODNESS_TIMELINE_PLOT)), eta_s=ETA_MIN["backtest"] * 60, optional=True))
if RUN_PUBLISH:
    STAGES.append(NotebookStage("publish", "Publish curated artifacts", eta_s=ETA_MIN["publish"] * 60, optional=True))


def show_stage_status():
    rows = stage_status_rows(MANIFEST_PATH, STAGES, root=ROOT)
    df = pd.DataFrame(rows)
    if not df.empty:
        display(df)
    print("Remaining ETA:", format_duration(remaining_eta_seconds(MANIFEST_PATH, STAGES, root=ROOT)))


print("run profile:", RUN_PROFILE)
print("run id:", RUN_ID)
print("run root:", RUN_ROOT)
print("base config:", BASE_CONFIG)
print("runtime config:", RUNTIME_CONFIG)
print("benchmark modes:", BENCHMARK_MODES)
print("graph source base:", GRAPH_PATH_FOR_RUN)
print("graph source resolved:", GRAPH_RESOLVED_PATH, f"(format={GRAPH_FORMAT})")
print("prices path:", DATA_PRICES_PATH)
show_stage_status()


In [ ]:
preflight_rows = [
    {"item": "run_profile", "value": RUN_PROFILE},
    {"item": "enabled_stages", "value": ", ".join(ENABLED_STAGE_NAMES)},
    {"item": "base_config", "value": str(BASE_CONFIG)},
    {"item": "runtime_config", "value": str(RUNTIME_CONFIG)},
    {"item": "graph_source_base", "value": str(GRAPH_PATH_FOR_RUN)},
    {"item": "graph_source_resolved", "value": str(GRAPH_RESOLVED_PATH)},
    {"item": "graph_format", "value": GRAPH_FORMAT},
    {"item": "graph_count", "value": GRAPH_COUNT},
    {"item": "graph_num_shards", "value": GRAPH_NUM_SHARDS},
    {"item": "graph_shard_size", "value": GRAPH_SHARD_SIZE},
    {"item": "benchmark_epochs", "value": BENCHMARK_EPOCHS},
    {"item": "benchmark_mode_count", "value": BENCHMARK_MODE_COUNT},
    {"item": "benchmark_modes", "value": ", ".join(BENCHMARK_MODES)},
    {"item": "benchmark_walk_forward_max_folds", "value": BENCHMARK_WALK_FORWARD_MAX_FOLDS},
    {"item": "benchmark_fold_estimate", "value": BENCHMARK_FOLD_ESTIMATE},
    {"item": "benchmark_work_units", "value": BENCHMARK_WORK_UNITS},
    {"item": "benchmark_checkpointing", "value": "per-mode tracked steps + epoch resume"},
    {"item": "train_resume_state", "value": str(TRAIN_RESUME_STATE)},
    {"item": "benchmark_resume_dir", "value": str(BENCHMARK_RESUME_DIR)},
]
display(pd.DataFrame(preflight_rows))

if GRAPH_FORMAT == "sharded":
    print("Sharded graph artifact will be used:", GRAPH_RESOLVED_PATH)
    if GRAPH_MANIFEST_PATH is not None:
        print("Sharded manifest:", GRAPH_MANIFEST_PATH)
elif SHARDED_GRAPH_PATH.is_dir():
    print("Sharded sidecar exists and will be preferred by the training/benchmark scripts:", SHARDED_GRAPH_PATH)
else:
    print("Using packed graph artifact:", GRAPH_RESOLVED_PATH)
    print(
        "Recommendation: run notebooks/shard_graph_artifact_colab.ipynb before large runs "
        "to reduce load latency and memory pressure."
    )

print("Notebook execution is split across stage cells; rerun only the cell you need.")
print("Epoch-level resume is enabled for train and benchmark runs.")
print("Train resume state:", TRAIN_RESUME_STATE)
print("Benchmark resume dir:", BENCHMARK_RESUME_DIR)
if RUN_BENCHMARK:
    print("Benchmark modes checkpoint independently:", ", ".join(BENCHMARK_MODES))

if BENCHMARK_WORK_UNITS is not None:
    print(
        "Rough benchmark work estimate:",
        f"{BENCHMARK_EPOCHS} epochs x {BENCHMARK_MODE_COUNT} modes x {BENCHMARK_FOLD_ESTIMATE} folds = {BENCHMARK_WORK_UNITS}",
    )
else:
    print("Rough benchmark work estimate unavailable because graph count could not be inferred.")


In [ ]:
required_modules = ["torch", "torch_geometric", "pandas", "numpy", "tqdm", "matplotlib"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if INSTALL_DEPS or missing:
    install_cmd = (
        f"{PYTHON_EXE} -m pip install --upgrade pip setuptools wheel && "
        f"{PYTHON_EXE} -m pip install -r requirements.txt && "
        f"{PYTHON_EXE} -m pip install -e ."
    )
    run_tracked_command(
        step="install",
        label="Install dependencies",
        command=install_cmd,
        manifest_path=MANIFEST_PATH,
        eta_s=ETA_MIN["install"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        metadata={"missing_modules": missing},
    )
else:
    print("Dependencies already installed. Skipping install stage.")

show_stage_status()


In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np


def run_python_step(
    *,
    step: str,
    label: str,
    runner,
    required_outputs=(),
    eta_s: float = 0.0,
    optional: bool = False,
    metadata: dict | None = None,
    skip_if_complete: bool = True,
):
    outputs = [Path(path) for path in required_outputs]
    if skip_if_complete and step_is_complete(MANIFEST_PATH, step, required_outputs=outputs):
        print(f"Skipping {label}: outputs already exist.")
        record_run_step(
            MANIFEST_PATH,
            step=step,
            command=f"python:{label}",
            status="completed",
            required_outputs=outputs,
            metadata={
                "skipped": True,
                "eta_s": float(eta_s),
                "optional": bool(optional),
                **(metadata or {}),
            },
        )
        return None

    record_run_step(
        MANIFEST_PATH,
        step=step,
        command=f"python:{label}",
        status="started",
        required_outputs=outputs,
        metadata={
            "eta_s": float(eta_s),
            "optional": bool(optional),
            **(metadata or {}),
        },
    )
    started = time.time()
    try:
        result = runner()
    except Exception as exc:
        elapsed_s = time.time() - started
        record_run_step(
            MANIFEST_PATH,
            step=step,
            command=f"python:{label}",
            status="failed",
            required_outputs=outputs,
            metadata={
                "elapsed_s": elapsed_s,
                "eta_s": float(eta_s),
                "optional": bool(optional),
                "error": str(exc),
                **(metadata or {}),
            },
        )
        raise

    elapsed_s = time.time() - started
    record_run_step(
        MANIFEST_PATH,
        step=step,
        command=f"python:{label}",
        status="completed",
        required_outputs=outputs,
        metadata={
            "elapsed_s": elapsed_s,
            "eta_s": float(eta_s),
            "optional": bool(optional),
            **(metadata or {}),
        },
    )
    print(f"Completed {label} in {elapsed_s:.2f}s")
    return result


def ensure_benchmark_mode_config(spec: dict) -> Path:
    mode_overrides = {
        "benchmark": {
            "out_csv": str(spec["out_csv"]),
            "walk_forward_out_csv": str(spec["folds_csv"]),
            "baseline_out_csv": str(spec["baseline_csv"]),
            "history_out_csv": str(spec["history_csv"]),
            "plot_path": str(spec["plot_path"]),
            "bar_plot_path": str(spec["bar_plot_path"]),
        }
    }
    return write_toml_overrides(RUNTIME_CONFIG, spec["config_path"], mode_overrides)


def run_benchmark_mode_stage(spec: dict):
    config_path = ensure_benchmark_mode_config(spec)
    command = (
        f"{PYTHON_EXE} scripts/benchmark_training.py "
        f"--config {shell_quote(str(config_path))} "
        f"--modes {shell_quote(spec['mode'])}"
    )
    return run_tracked_command(
        step=spec["step"],
        label=f"Benchmark {spec['mode']}",
        command=command,
        manifest_path=MANIFEST_PATH,
        required_outputs=[spec["out_csv"], spec["history_csv"], spec["plot_path"], spec["bar_plot_path"]],
        eta_s=BENCHMARK_MODE_ETA_S,
        log_dir=LOG_DIR,
        cwd=ROOT,
        metadata={"mode": spec["mode"], "config_path": str(config_path)},
    )


def plot_merged_benchmark_outputs(benchmark_csv: Path, plot_path: Path, bar_plot_path: Path) -> None:
    bench_df = pd.read_csv(benchmark_csv)
    plot_rows = bench_df.copy()
    if "status" in plot_rows.columns:
        plot_rows = plot_rows[plot_rows["status"].astype(str).str.strip().str.lower() == "ok"]
    if "row_type" in plot_rows.columns:
        plot_rows = plot_rows[plot_rows["row_type"].astype(str).str.strip().str.lower() != "seed"]
    plot_rows["graphs_per_s"] = pd.to_numeric(plot_rows.get("graphs_per_s"), errors="coerce")
    plot_rows["primary_eval_metric"] = pd.to_numeric(plot_rows.get("primary_eval_metric"), errors="coerce")
    plot_rows = plot_rows[np.isfinite(plot_rows["graphs_per_s"])].copy()
    if plot_rows.empty:
        raise RuntimeError("no successful benchmark rows available for plotting")

    families = []
    seen = set()
    for family in plot_rows.get("task_family", pd.Series(["custom"] * len(plot_rows))).astype(str):
        family = family.strip() or "custom"
        if family not in seen:
            seen.add(family)
            families.append(family)

    fig, axes = plt.subplots(len(families), 1, figsize=(7, 4 * max(1, len(families))))
    if len(families) == 1:
        axes = [axes]
    for ax, family in zip(axes, families):
        family_rows = plot_rows[plot_rows.get("task_family", "custom").astype(str).str.strip() == family]
        xs = family_rows["graphs_per_s"].astype(float).tolist()
        ys = pd.to_numeric(family_rows.get("primary_eval_metric"), errors="coerce").astype(float).tolist()
        labels = family_rows.get("mode", pd.Series([""] * len(family_rows))).astype(str).tolist()
        metric_names = sorted({str(v).strip() for v in family_rows.get("primary_eval_metric_name", pd.Series(dtype=str)).astype(str).tolist() if str(v).strip()})
        ax.scatter(xs, ys, color="#4C78A8")
        for x, y, label in zip(xs, ys, labels):
            ax.annotate(label, (x, y), textcoords="offset points", xytext=(6, 4))
        ax.set_xlabel("graphs/sec")
        ax.set_ylabel("primary metric")
        title_suffix = ", ".join(metric_names) if metric_names else "primary metric"
        ax.set_title(f"{family}: speed vs {title_suffix}")
    fig.tight_layout()
    plot_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)

    fig, axes = plt.subplots(len(families), 2, figsize=(12, 4 * max(1, len(families))))
    if len(families) == 1:
        axes = np.asarray([axes], dtype=object)
    for row_axes, family in zip(axes, families):
        family_rows = plot_rows[plot_rows.get("task_family", "custom").astype(str).str.strip() == family]
        modes_family = family_rows.get("mode", pd.Series([""] * len(family_rows))).astype(str).tolist()
        epoch_s_family = pd.to_numeric(family_rows.get("avg_epoch_s"), errors="coerce").astype(float).tolist()
        quality_family = pd.to_numeric(family_rows.get("primary_eval_metric"), errors="coerce").astype(float).tolist()
        row_axes[0].bar(modes_family, epoch_s_family, color="#4C78A8")
        row_axes[0].set_title(f"{family}: avg epoch time")
        row_axes[0].set_ylabel("seconds")
        row_axes[1].bar(modes_family, quality_family, color="#72B7B2")
        row_axes[1].set_title(f"{family}: primary metric")
        row_axes[1].set_ylabel("value")
    fig.tight_layout()
    bar_plot_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(bar_plot_path, dpi=150)
    plt.close(fig)


def aggregate_benchmark_mode_outputs(required: bool = True):
    specs = BENCHMARK_MODE_SPECS
    missing = [spec["mode"] for spec in specs if not spec["out_csv"].exists()]
    if required and missing:
        raise FileNotFoundError("Missing per-mode benchmark outputs for: " + ", ".join(missing))

    available_specs = [spec for spec in specs if spec["out_csv"].exists()]
    if not available_specs:
        raise RuntimeError("No per-mode benchmark outputs available to merge.")

    def _runner():
        merge_csv_files([spec["out_csv"] for spec in available_specs], BENCHMARK_CSV)
        merge_csv_files([spec["folds_csv"] for spec in available_specs], BENCHMARK_FOLDS_CSV)
        merge_csv_files([spec["baseline_csv"] for spec in available_specs], BENCHMARK_BASELINE_CSV)
        merge_csv_files([spec["history_csv"] for spec in available_specs], BENCHMARK_HISTORY_CSV)
        plot_merged_benchmark_outputs(BENCHMARK_CSV, BENCHMARK_PLOT, BENCHMARK_BAR)
        print("Merged benchmark modes:", ", ".join(spec["mode"] for spec in available_specs))
        if missing:
            print("Modes still missing from merge:", ", ".join(missing))

    return run_python_step(
        step="benchmark_aggregate",
        label="Merge benchmark outputs",
        runner=_runner,
        required_outputs=[BENCHMARK_CSV, BENCHMARK_FOLDS_CSV, BENCHMARK_BASELINE_CSV, BENCHMARK_HISTORY_CSV, BENCHMARK_PLOT, BENCHMARK_BAR],
        eta_s=ETA_MIN["benchmark_aggregate"] * 60,
        metadata={"modes": BENCHMARK_MODES},
    )


In [ ]:
if RUN_BUILD_GRAPHS:
    run_tracked_command(
        step="build_graphs",
        label="Build graphs",
        command=f"{PYTHON_EXE} scripts/build_graphs.py --config {shell_quote(str(RUNTIME_CONFIG))}",
        manifest_path=MANIFEST_PATH,
        required_outputs=[GRAPH_OUT],
        eta_s=ETA_MIN["build_graphs"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )
else:
    if not GRAPH_RESOLVED_PATH.exists():
        raise FileNotFoundError(f"Missing graph artifact: {GRAPH_RESOLVED_PATH}")
    print("Using existing graph artifact:", GRAPH_RESOLVED_PATH, f"(format={GRAPH_FORMAT})")

show_stage_status()


In [ ]:
if RUN_BENCHMARK:
    for spec in BENCHMARK_MODE_SPECS:
        print(f"\n=== benchmark mode: {spec['mode']} ===")
        run_benchmark_mode_stage(spec)

    aggregate_benchmark_mode_outputs(required=True)

    run_tracked_command(
        step="paper_summary",
        label="Write paper benchmark summary",
        command=(
            f"{PYTHON_EXE} scripts/paper_benchmark_summary.py "
            f"--benchmark {shell_quote(str(BENCHMARK_CSV))} "
            f"--folds-csv {shell_quote(str(BENCHMARK_FOLDS_CSV))} "
            f"--out-md {shell_quote(str(PAPER_SUMMARY_MD))} "
            f"--out-csv {shell_quote(str(PAPER_SUMMARY_CSV))} "
            f"--out-json {shell_quote(str(PAPER_SUMMARY_JSON))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[PAPER_SUMMARY_MD, PAPER_SUMMARY_CSV, PAPER_SUMMARY_JSON],
        eta_s=ETA_MIN["paper_summary"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )
else:
    print("Skipping benchmark stage.")

show_stage_status()


In [ ]:
if RUN_TRAIN:
    run_tracked_command(
        step="train",
        label="Train FF/BP model",
        command=f"{PYTHON_EXE} scripts/train_ff_gnn.py --config {shell_quote(str(RUNTIME_CONFIG))}",
        manifest_path=MANIFEST_PATH,
        required_outputs=[MODEL_CKPT, ENCODER_CKPT, CRITIC_CKPT, TRAIN_LOG_CSV],
        eta_s=ETA_MIN["train"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )
else:
    print("Skipping train stage.")

show_stage_status()


In [ ]:
if RUN_SWEEP:
    run_tracked_command(
        step="sweep",
        label="Run FF sweep",
        command=f"{PYTHON_EXE} scripts/ff_sweep.py --config {shell_quote(str(RUNTIME_CONFIG))}",
        manifest_path=MANIFEST_PATH,
        required_outputs=[SWEEP_CSV],
        eta_s=ETA_MIN["sweep"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )
else:
    print("Skipping sweep stage.")

if RUN_DUAL_SCORE and RUN_BENCHMARK and RUN_SWEEP:
    run_tracked_command(
        step="dual_score",
        label="Write dual score report",
        command=(
            f"{PYTHON_EXE} scripts/dual_score_report.py "
            f"--benchmark {shell_quote(str(BENCHMARK_CSV))} "
            f"--sweep {shell_quote(str(SWEEP_CSV))} "
            f"--out {shell_quote(str(DUAL_SCORE_TXT))} "
            f"--out-csv {shell_quote(str(DUAL_SCORE_CSV))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[DUAL_SCORE_TXT, DUAL_SCORE_CSV],
        eta_s=ETA_MIN["dual_score"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )
else:
    print("Skipping dual score stage.")

show_stage_status()


In [ ]:
if RUN_SCENARIO and RUN_TRAIN:
    run_tracked_command(
        step="scenario",
        label="Run scenario book",
        command=(
            f"{PYTHON_EXE} scripts/scenario_book.py "
            f"--config {shell_quote(str(RUNTIME_CONFIG))} "
            f"--critic-model {shell_quote(str(CRITIC_CKPT))} "
            f"--out {shell_quote(str(SCENARIO_CSV))} "
            f"--diag-out {shell_quote(str(SCENARIO_DIAG_CSV))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[SCENARIO_CSV, SCENARIO_DIAG_CSV],
        eta_s=ETA_MIN["scenario"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )
    run_tracked_command(
        step="stress",
        label="Write stress report",
        command=(
            f"{PYTHON_EXE} scripts/stress_test_report.py "
            f"--csv {shell_quote(str(SCENARIO_CSV))} "
            f"--out-csv {shell_quote(str(STRESS_CSV))} "
            f"--out-plot {shell_quote(str(STRESS_PLOT))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[STRESS_CSV, STRESS_PLOT],
        eta_s=ETA_MIN["stress"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )
    run_tracked_command(
        step="calibration",
        label="Write hallucination calibration",
        command=(
            f"{PYTHON_EXE} scripts/hallucination_calibration.py "
            f"--csv {shell_quote(str(SCENARIO_CSV))} "
            f"--out {shell_quote(str(CALIBRATION_JSON))} "
            f"--out-by-ticker {shell_quote(str(CALIBRATION_BY_TICKER_CSV))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[CALIBRATION_JSON, CALIBRATION_BY_TICKER_CSV],
        eta_s=ETA_MIN["calibration"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )
else:
    print("Skipping scenario/stress/calibration stages.")

show_stage_status()


In [ ]:
if RUN_BACKTEST and RUN_TRAIN:
    run_tracked_command(
        step="backtest",
        label="Run goodness backtest",
        command=(
            f"{PYTHON_EXE} scripts/goodness_backtest.py "
            f"--config {shell_quote(str(RUNTIME_CONFIG))} "
            f"--prices {shell_quote(str(DATA_PRICES_PATH))} "
            f"--out-csv {shell_quote(str(GOODNESS_CSV))} "
            f"--out-quantiles {shell_quote(str(GOODNESS_QUANTILES_CSV))} "
            f"--out-plot {shell_quote(str(GOODNESS_PLOT))} "
            f"--out-events {shell_quote(str(GOODNESS_EVENTS_CSV))} "
            f"--out-strategy {shell_quote(str(GOODNESS_STRATEGY_CSV))} "
            f"--out-timeline {shell_quote(str(GOODNESS_TIMELINE_PLOT))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[GOODNESS_CSV, GOODNESS_STRATEGY_CSV, GOODNESS_TIMELINE_PLOT],
        eta_s=ETA_MIN["backtest"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )
else:
    print("Skipping backtest stage.")

if RUN_PUBLISH:
    run_tracked_command(
        step="publish",
        label="Publish curated artifacts",
        command=f"{PYTHON_EXE} scripts/publish_run.py --run-id {shell_quote(RUN_ID)}",
        manifest_path=MANIFEST_PATH,
        eta_s=ETA_MIN["publish"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )
else:
    print("Skipping publish stage.")

show_stage_status()


In [ ]:
ARTIFACTS = [
    RUNTIME_CONFIG,
    GRAPH_PATH_FOR_RUN,
    TRAIN_LOG_CSV,
    TRAIN_PLOT,
    MODEL_CKPT,
    ENCODER_CKPT,
    CRITIC_CKPT,
    TRAIN_RESUME_STATE,
    BENCHMARK_RESUME_DIR,
    BENCHMARK_CSV,
    BENCHMARK_FOLDS_CSV,
    BENCHMARK_BASELINE_CSV,
    BENCHMARK_HISTORY_CSV,
    BENCHMARK_PLOT,
    BENCHMARK_BAR,
    PAPER_SUMMARY_MD,
    PAPER_SUMMARY_CSV,
    PAPER_SUMMARY_JSON,
    SWEEP_CSV,
    DUAL_SCORE_TXT,
    DUAL_SCORE_CSV,
    SCENARIO_CSV,
    SCENARIO_DIAG_CSV,
    STRESS_CSV,
    STRESS_PLOT,
    CALIBRATION_JSON,
    CALIBRATION_BY_TICKER_CSV,
    GOODNESS_CSV,
    GOODNESS_QUANTILES_CSV,
    GOODNESS_PLOT,
    GOODNESS_EVENTS_CSV,
    GOODNESS_STRATEGY_CSV,
    GOODNESS_TIMELINE_PLOT,
    MANIFEST_PATH,
]
for spec in BENCHMARK_MODE_SPECS:
    ARTIFACTS.extend([
        spec["config_path"],
        spec["out_csv"],
        spec["folds_csv"],
        spec["baseline_csv"],
        spec["history_csv"],
        spec["plot_path"],
        spec["bar_plot_path"],
    ])

rows = []
for path in ARTIFACTS:
    rows.append({
        "path": str(path),
        "exists": path.exists(),
        "bytes": path.stat().st_size if path.exists() and path.is_file() else None,
    })

display(pd.DataFrame(rows))
show_stage_status()

if PAPER_SUMMARY_MD.exists():
    print("\nPaper summary markdown path:", PAPER_SUMMARY_MD)
if MANIFEST_PATH.exists():
    print("Stage manifest:", MANIFEST_PATH)
    print(MANIFEST_PATH.read_text()[:4000])
